# Data Gathering

## Sumber Data

Proyek ini menggunakan dua dataset:

1. **Dataset Komposisi Pangan** : diperoleh melalui web scraping dari website [panganku.org](https://www.panganku.org/id-ID/), yaitu website resmi Data Komposisi Pangan Indonesia yang dikelola oleh Direktorat Gizi Masyarakat, Kementerian Kesehatan RI. Website ini memuat data kandungan nutrisi berbagai bahan makanan Indonesia berdasarkan Tabel Komposisi Pangan Indonesia (TKPI).

2. **Dataset AKG Anak** : dibuat secara manual berdasarkan data Angka Kecukupan Gizi (AKG) 2019 yang ditetapkan melalui Peraturan Menteri Kesehatan RI No. 28 Tahun 2019, berisi kebutuhan gizi harian anak usia 0-6 tahun.

## Pengecekan Robots.txt

Sebelum melakukan scraping, dilakukan pengecekan file `robots.txt` pada panganku.org untuk memastikan proses scraping diizinkan secara etis sesuai kebijakan website.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'id-ID,id;q=0.9',
})

# Cek robots.txt
r_robots = session.get('https://www.panganku.org/robots.txt')
print('=== robots.txt ===')
print(r_robots.text)

=== robots.txt ===
User-agent: *
Allow: /


Hasil pengecekan menunjukkan `Allow: /` yang berarti seluruh user-agent diizinkan untuk mengakses website. Proses scraping dapat dilakukan.

## Scraping Dataset Komposisi Pangan

Proses scraping dilakukan dalam 3 tahap:
1. Mengambil seluruh kode pangan dari halaman `semua_nutrisi`
2. Melakukan POST request ke halaman `view` untuk setiap kode pangan
3. Menyimpan hasil ke file `panganku_dataset.csv`

Throttling diterapkan sebesar 0.5 detik per request untuk menghindari rate limiting.

In [2]:
# Step 1 - hit homepage untuk dapat cookies
r = session.get('https://www.panganku.org/id-ID/')
print('Homepage status:', r.status_code)

# Step 2 - ambil semua kode makanan
r_list = session.get('https://www.panganku.org/id-ID/semua_nutrisi')
soup_list = BeautifulSoup(r_list.text, 'html.parser')
table = soup_list.find('table')
kode_list = []
for tr in table.find_all('tr')[1:]:
    tds = tr.find_all('td')
    if tds:
        kode_list.append(tds[1].text.strip())

print(f'Total kode makanan: {len(kode_list)}')
print('Contoh kode:', kode_list[:5])

Homepage status: 200
Total kode makanan: 1146
Contoh kode: ['GP053', 'GP054', 'DR001', 'DR002', 'ER001']


In [ ]:
# Step 3 - scrape detail tiap makanan
def parse_detail(html):
    soup = BeautifulSoup(html, 'html.parser')
    data = {}
    for table in soup.find_all('table'):
        for row in table.find_all('tr'):
            tds = row.find_all('td')
            if len(tds) >= 2:
                key = tds[0].text.strip().replace('\n', '').replace(':', '').strip()
                val = tds[-1].text.strip().replace('\n', '').strip()
                if key:
                    data[key] = val
    return data

all_data = []
failed_kode = []
for i, kode in enumerate(kode_list):
    try:
        r = session.post('https://www.panganku.org/id-ID/view', data={'haha': kode})
        data = parse_detail(r.text)
        all_data.append(data)
        if (i+1) % 50 == 0:
            print(f'Progress: {i+1}/{len(kode_list)}')
        time.sleep(0.5)
    except Exception as e:
        print(f'Error di {kode}: {e}')
        failed_kode.append(kode)

print(f'\nSelesai! Total berhasil: {len(all_data)} | Gagal: {len(failed_kode)}')
if failed_kode:
    print(f'Kode yang gagal: {failed_kode}')

df_pangan = pd.DataFrame(all_data)
df_pangan.to_csv('panganku_dataset.csv', index=False)
print(f'\nSelesai! Total {len(df_pangan)} makanan tersimpan.')
print(df_pangan.head())

Progress: 50/1146
Progress: 100/1146
Progress: 150/1146
Progress: 200/1146
Progress: 250/1146
Progress: 300/1146
Progress: 350/1146
Progress: 400/1146
Progress: 450/1146
Progress: 500/1146
Progress: 550/1146
Progress: 600/1146
Progress: 650/1146
Progress: 700/1146
Progress: 750/1146
Progress: 800/1146
Progress: 850/1146
Progress: 900/1146
Progress: 950/1146
Progress: 1000/1146
Progress: 1050/1146
Progress: 1100/1146

Selesai! Total 1146 makanan tersimpan.
    Kode                                     Nama          Nama Latin  \
0  GP053                             Abon haruwan                       
1  GP054                                Abon ikan                       
2  DR001  Akar tonjong, segar (Lotus root, fresh)    Nelumbo nucifera   
3  DR002                          Ale,toge, segar  Acacia leucophloea   
4  ER001          Alpukat, segar (Avocado, fresh)    Persea americana   

                         Asal               Kategori          Tipe Bahan  \
0  Daerah Istimewa Yogyak

## Membuat Dataset AKG Anak

Dataset AKG dibuat secara manual berdasarkan Peraturan Menteri Kesehatan RI No. 28 Tahun 2019 tentang Angka Kecukupan Gizi yang Dianjurkan untuk Masyarakat Indonesia. Data mencakup kebutuhan gizi harian anak usia 0-6 tahun.

In [4]:
# Membuat dataset AKG Anak usia 0-6 tahun
# Sumber: Permenkes RI No. 28 Tahun 2019

data_akg = {
    'kelompok_umur': ['0-5 bulan', '6-11 bulan', '1-3 tahun', '4-6 tahun'],
    'energi_kal': [550, 800, 1350, 1400],
    'protein_g': [12, 18, 20, 25],
    'lemak_g': [34, 35, 45, 50],
    'karbohidrat_g': [58, 82, 215, 220],
    'serat_g': [0, 11, 19, 20],
    'air_ml': [700, 900, 1150, 1450],
    'kalsium_mg': [200, 270, 650, 1000],
    'fosfor_mg': [100, 275, 500, 500],
    'besi_mg': [0.3, 11, 7, 10],
    'natrium_mg': [120, 370, 800, 1000],
    'kalium_mg': [400, 700, 2000, 2300],
    'seng_mg': [2, 3, 4, 5],
    'retinol_mcg': [375, 400, 400, 450],
    'thiamin_mg': [0.3, 0.4, 0.6, 0.8],
    'riboflavin_mg': [0.3, 0.4, 0.7, 1.0],
    'niasin_mg': [3, 5, 8, 9],
    'vit_c_mg': [40, 50, 40, 45],
}

df_akg = pd.DataFrame(data_akg)
df_akg.to_csv('akg_anak.csv', index=False)
print('Berhasil disimpan ke akg_anak.csv!')
print(df_akg.to_string())

Berhasil disimpan ke akg_anak.csv!
  kelompok_umur  energi_kal  protein_g  lemak_g  karbohidrat_g  serat_g  air_ml  kalsium_mg  fosfor_mg  besi_mg  natrium_mg  kalium_mg  seng_mg  retinol_mcg  thiamin_mg  riboflavin_mg  niasin_mg  vit_c_mg
0     0-5 bulan         550         12       34             58        0     700         200        100      0.3         120        400        2          375         0.3            0.3          3        40
1    6-11 bulan         800         18       35             82       11     900         270        275     11.0         370        700        3          400         0.4            0.4          5        50
2     1-3 tahun        1350         20       45            215       19    1150         650        500      7.0         800       2000        4          400         0.6            0.7          8        40
3     4-6 tahun        1400         25       50            220       20    1450        1000        500     10.0        1000       2300        5  

## Hasil Gathering

Proses pengumpulan data telah selesai dilakukan. Berikut ringkasan dataset yang diperoleh:

| Dataset | Sumber | Jumlah Data | File |
|---|---|---|---|
| Komposisi Pangan | panganku.org (web scraping) | 1.146 bahan makanan, 28 kolom | `panganku_dataset.csv` |
| AKG Anak | Permenkes RI No. 28 Tahun 2019 | 4 kelompok umur, 18 kolom | `akg_anak.csv` |

Kedua dataset ini akan digunakan pada notebook analisis (`02_Analisis.ipynb`) untuk proses data wrangling, EDA, dan visualisasi.